# **Trabalho 2.1: Matemática Computacional**
---

In [ ]:
import math
import time
import random
import pandas as pd
from IPython.display import display, Markdown

# configuração para exibição no pandas
pd.set_option('display.float_format', '{:.8e}'.format)

def formatar_cientifico(valor, precisao=5):
    """Formata float para notação científica amigável (ex: 1.23 x 10^-4)."""
    if valor == 0: return "0.0"
    s = "{:.{}e}".format(valor, precisao)
    base, expoente = s.split('e')
    if expoente.startswith('+'): expoente = expoente[1:]
    return f"{base} x 10^{expoente}"

def gerar_tabelas_comparativas(resultados_dict):
    """Gera DataFrames de Resultados, Esforço e Tempo para comparação."""
    dados_num, dados_esf, dados_tem = {}, {}, {}

    for metodo, res in resultados_dict.items():
        dados_num[metodo], dados_esf[metodo], dados_tem[metodo] = res

    # Exibição organizada
    display(Markdown("### Resultados Numéricos"))
    display(pd.DataFrame(dados_num, index=["Dados Iniciais", "Raiz (x̄)", "f(x̄)", "Erro Final", "Iterações"]))
    
    display(Markdown("### Análise de Esforço Computacional"))
    display(pd.DataFrame(dados_esf, index=["Ops Aritméticas/Iter", "Complexidade", "Decisões Lógicas Totais", "Avaliações de Função/Iter", "Total Iterações"]))
    
    display(Markdown("### Análise de Tempo"))
    display(pd.DataFrame(dados_tem, index=["Tempo/Iteração (ms)", "Tempo Total (ms)"]))

## **Questão A**

### **Funções do DataFrame usadas nos exemplos 18 - 21**:

In [56]:
def format_scientific(value, precision=4):
    if value == 0: return "0.0000 x 10^0"
    formatted = "{:.{}e}".format(value, precision)
    return formatted.replace("e", " x 10^").replace("+", "")

def get_numerical_results_table(numerical_data):
    return pd.DataFrame({
        "Bisseção": numerical_data["bisection"],
        "Falsa Posição": numerical_data["false_position"],
        "Ponto Fixo": numerical_data["fixed_point"],
        "Newton": numerical_data["newton"],
        "Secante": numerical_data["secant"]
    }, index=[
        "Dados Iniciais", "x̄", "f(x̄)", "Erro em x", "Número de Iterações"
    ])

def get_computational_effort_table(effort_data):
    return pd.DataFrame({
        "Bisseção": effort_data["bisection"],
        "Falsa Posição": effort_data["false_position"],
        "Ponto Fixo": effort_data["fixed_point"],
        "Newton": effort_data["newton"],
        "Secante": effort_data["secant"]
    }, index=[
        "Operações por Iteração", "Complexidade de Operação", 
        "Decisões Lógicas Totais", "Avaliações de Função por Iteração", 
        "Número de Iterações"
    ])

def get_execution_time_table(time_data):
    return pd.DataFrame({
        "Bisseção": time_data["bisection"],
        "Falsa Posição": time_data["false_position"],
        "Ponto Fixo": time_data["fixed_point"],
        "Newton": time_data["newton"],
        "Secante": time_data["secant"]
    }, index=[
        "Tempo por Iteração (ms)", "Tempo Total (ms)"
    ])

def print_tables(numerical_data, effort_data, time_data):
    display(Markdown("## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes"))
    display(get_numerical_results_table(numerical_data))

    display(Markdown("## Tabela 2 – Análise de Esforço Computacional"))
    display(get_computational_effort_table(effort_data))

    display(Markdown("## Tabela 3 – Análise de Tempo de Execução"))
    display(get_execution_time_table(time_data))

### **Função do método de Bisseção**

#### O método da Bisseção funciona da seguinte maneira:

<div style="text-align: center;">
    <img src="bisection.png" width="600">
</div>

Ele escolhe 2 pontos no eixo X (A e B), calcula a média entre os 2 pontos, armazena numa variável x e então calcula f(x).

Após isso ele verifica o sinal de f(x), se for negativo, então f(x) está mais perto de f(A), logo, o ponto A é substituído pelo valor da variável x.

Se o valor de f(x) for positivo, então f(x) está mais perto de f(B), logo, o ponto B é substituído pelo valor da variável x.

Esse processo é repetido ate a diferença entre A e B ou o valor de f(x) ser muito pequeno.

In [ ]:
def bisection_method(function, interval, stopping_crit, precision=4, max_iterations=100):
    a, b = interval
    epsilon = stopping_crit
    x = 0
    iterations, total_time = 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0
    
    # verificação inicial
    dyn_ops += 1; dyn_logic += 1
    if (b - a) < epsilon:
        x = random.uniform(a, b)
    else:
        iterations = 1
        start_time = time.perf_counter()
        for i in range(max_iterations):
            dyn_evals += 1; fa = function(a)
            dyn_ops += 2; x = (a + b) / 2
            dyn_evals += 1; fx = function(x)
            
            dyn_ops += 1; dyn_logic += 1
            if fa * fx > 0:
                a = x
            else:
                b = x
            
            dyn_ops += 1; dyn_logic += 1
            if abs(b - a) < epsilon:
                break
            iterations += 1
        
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 # em ms

    # retorna as 3 listas: numerica, esforço, tempo
    numerical = [str(interval), x, format_scientific(function(x), precision), format_scientific(abs(b-a), precision), iterations]
    effort = [dyn_ops // iterations if iterations > 0 else dyn_ops, "O(1)", dyn_logic, dyn_evals // iterations if iterations > 0 else dyn_evals, iterations]
    time_exec = [total_time / iterations if iterations > 0 else 0, total_time]
    
    return numerical, effort, time_exec

### **Função do método de Posição Falsa**

#### O método da Posição Falsa funciona da seguinte maneira:

<div style="text-align: center;">
    <img src="false_position.png" width="600">
</div>

Ele escolhe 2 pontos no eixo X (A e B), calcula f(A) e f(B) e traça uma reta entre esses 2 pontos.

Depois, ele verifica qual o valor x dessa reta que passa pelo y = 0, calculando f(x) logo em seguida.

Após isso, assim como no método de Bisseção, ele verifica o sinal de f(x) e substitui A ou B dependendo do sinal de f(x).

Esse processo é repetido ate a diferença entre A e B ou o valor de f(x) ser muito pequeno.

In [ ]:
def false_position_method(function, interval, precisions, precision_format=4, max_iterations=100):
    a, b = interval
    epsilon1, epsilon2 = precisions
    x = 0
    iterations, total_time = 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    # verificação inicial
    dyn_ops += 1; dyn_logic += 1
    if (b - a) < epsilon1:
        x = random.uniform(a, b)
    else:
        dyn_evals += 1; dyn_logic += 1
        if abs(function(a)) < epsilon2: x = a
        elif abs(function(b)) < epsilon2: 
            dyn_evals += 1; dyn_logic += 1; x = b
        else:
            iterations = 1
            start_time = time.perf_counter()
            for i in range(max_iterations):
                dyn_evals += 2; fa = function(a); fb = function(b)
                dyn_ops += 5; x = ((a * fb) - (b * fa)) / (fb - fa)
                dyn_evals += 1; fx = function(x)
                
                dyn_logic += 1
                if abs(fx) < epsilon2: break
                
                dyn_ops += 1; dyn_logic += 1
                if fa * fx > 0: a = x
                else: b = x
                
                dyn_ops += 1; dyn_logic += 1
                if abs(b - a) < epsilon1: break
                iterations += 1
            
            final_time = time.perf_counter()
            total_time = (final_time - start_time) * 1000

    numerical = [str(interval), x, format_scientific(function(x), precision_format), format_scientific(abs(b-a), precision_format), iterations]
    effort = [dyn_ops // iterations if iterations > 0 else dyn_ops, "O(1)", dyn_logic, dyn_evals // iterations if iterations > 0 else dyn_evals, iterations]
    time_exec = [total_time / iterations if iterations > 0 else 0, total_time]
    
    return numerical, effort, time_exec

### **Função do método do Ponto Fixo**

#### O método do Ponto Fixo funciona da seguinte maneira:

<div style="text-align: center;">
    <img src="fixed_point.png" width="600">
</div>

Ele primeiro gera uma Função de Iteração $\phi(x)$ da função original, junto de uma função auxiliar y = x.

Onde essas duas funções de encontram, é onde está a raiz da função original.

Assim, ele dá um chute X inicial e calcula $\phi(X)$, após isso, ele pega um novo X1 = $\phi(X)$.

Esse processo é repetido ate a diferença entre Xn e Xn-1 ou o valor de f(Xn) ser muito pequeno.

In [59]:
def fixed_point_method(function, iteration_function, init_x, precisions, precision_format=4, max_iterations=100):
    x = init_x
    epsilon1, epsilon2 = precisions
    x_1, iterations, total_time, current_error = 0, 0, 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    dyn_evals += 1; dyn_logic += 1
    if abs(function(x)) < epsilon1: pass
    else:
        iterations = 1
        start_time = time.perf_counter()
        for i in range(max_iterations):
            dyn_evals += 1; x_1 = iteration_function(x)
            dyn_evals += 1; fx_1 = function(x_1)
            dyn_ops += 1; current_error = x_1 - x
            
            dyn_logic += 2
            if abs(fx_1) < epsilon1 or abs(current_error) < epsilon2:
                x = x_1; break
            
            x = x_1
            iterations += 1
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000

    numerical = [f"X0 = {init_x}", x, format_scientific(function(x), precision_format), format_scientific(abs(current_error), precision_format), iterations]
    effort = [dyn_ops // iterations if iterations > 0 else dyn_ops, "O(1)", dyn_logic, dyn_evals // iterations if iterations > 0 else dyn_evals, iterations]
    time_exec = [total_time / iterations if iterations > 0 else 0, total_time]
    
    return numerical, effort, time_exec

### **Função do método de Newton-Raphson**

#### O método de Newton-Raphson funciona da seguinte maneira:

<div style="text-align: center;">
    <img src="newton.png" width="600">
</div>

Ele dá um chute X inicial e calcula a tangente da função em f(X).

Após isso ele verifica qual o X1 que faz a reta tangente ser y = 0 e então usa esse X1 como novo X.

Esse processo é repetido ate a diferença entre Xn e Xn-1 ou o valor de f(Xn) ser muito pequeno.

In [ ]:
def newton_method(function, derivative_function, init_x, precisions, precision_format=4, max_iterations=100):
    x = init_x
    epsilon1, epsilon2 = precisions
    iterations, total_time, current_error = 0, 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    # se a derivada não for passada, assume lógica interna ou erro. 
    # vamos assumir que é possivel chamar derivative_functio
    
    dyn_evals += 1; dyn_logic += 1
    if abs(function(x)) < epsilon1: pass
    else:
        iterations = 1
        start_time = time.perf_counter()
        for i in range(max_iterations):
            dyn_evals += 2; # 1 f(x), 1 f'(x)
            f_val = function(x)
            df_val = derivative_function(x)
            
            dyn_ops += 2
            x_1 = x - (f_val / df_val)
            
            dyn_evals += 1; fx_1 = function(x_1)
            dyn_ops += 1; current_error = x_1 - x
            
            dyn_logic += 2
            if abs(fx_1) < epsilon1 or abs(current_error) < epsilon2:
                x = x_1; break
            
            x = x_1
            iterations += 1
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000

    numerical = [f"X0 = {init_x}", x, format_scientific(function(x), precision_format), format_scientific(abs(current_error), precision_format), iterations]
    effort = [dyn_ops // iterations if iterations > 0 else dyn_ops, "O(1)", dyn_logic, dyn_evals // iterations if iterations > 0 else dyn_evals, iterations]
    time_exec = [total_time / iterations if iterations > 0 else 0, total_time]
    
    return numerical, effort, time_exec

### **Função do método da Secante**

#### O método da Secante funciona da seguinte maneira:

<div style="text-align: center;">
    <img src="secant.png" width="600">
</div>

Ele dá um chute X0 e X1 inicial e calcula a secante entre f(X0) e f(X1).

Após isso ele verifica qual o X2 que faz a reta secante ser y = 0 e então substitui X0 = X1 e X1 = X2.

Esse processo é repetido ate a diferença entre Xn e Xn-1 ou o valor de f(Xn) ser muito pequeno.

In [61]:
def secant_method(function, initial_approximations, precisions, precision_format=4, max_iterations=100):
    x_0, x_1 = initial_approximations
    epsilon1, epsilon2 = precisions
    x = x_1
    iterations, total_time, current_error = 0, 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    dyn_evals += 1; dyn_logic += 1
    if abs(function(x_0)) < epsilon1: x = x_0
    else:
        dyn_evals += 1; dyn_ops += 1; dyn_logic += 2
        if abs(function(x_1)) < epsilon1 or abs(x_1 - x_0) < epsilon2: x = x_1
        else:
            iterations = 1
            start_time = time.perf_counter()
            for i in range(max_iterations):
                dyn_evals += 2; fx_0 = function(x_0); fx_1 = function(x_1)
                dyn_ops += 5; x_2 = x_1 - ((fx_1 / (fx_1 - fx_0)) * (x_1 - x_0))
                
                dyn_evals += 1; fx_2 = function(x_2)
                dyn_ops += 1; current_error = x_2 - x_1
                
                dyn_logic += 2
                if abs(fx_2) < epsilon1 or abs(current_error) < epsilon2:
                    x = x_2; break
                
                x_0 = x_1; x_1 = x_2
                iterations += 1
            final_time = time.perf_counter()
            total_time = (final_time - start_time) * 1000

    numerical = [f"X0 = {initial_approximations[0]}; X1 = {initial_approximations[1]}", x, format_scientific(function(x), precision_format), format_scientific(abs(current_error), precision_format), iterations]
    effort = [dyn_ops // iterations if iterations > 0 else dyn_ops, "O(1)", dyn_logic, dyn_evals // iterations if iterations > 0 else dyn_evals, iterations]
    time_exec = [total_time / iterations if iterations > 0 else 0, total_time]
    
    return numerical, effort, time_exec

## Exemplos Questão A

### **Exemplo 18**:

**Função Original:** $f(x) = e^{-x^2} - \cos(x)$

**Intervalo da Raiz:** $\xi \in (1, 2)$

**Precisão:** $\epsilon_1 = \epsilon_2 = 10^{-4}$

**Função de Iteração (Ponto Fixo):** $\phi(x) = \cos(x) - e^{-x^2} + x$

In [ ]:
# inicialização local dos dados
numerical_data, effort_data, time_data = {}, {}, {}

# definições
f_ex18 = lambda x: (math.e**(-x**2)) - math.cos(x)
phi_ex18 = lambda x: math.cos(x) - math.e**(-x**2) + x
interval_ex18 = [1, 2]
stopping_crit = 10**-4

# execuções
numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = \
    bisection_method(f_ex18, interval_ex18, stopping_crit, 4)

numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = \
    false_position_method(f_ex18, interval_ex18, (stopping_crit, stopping_crit), 4)

numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = \
    fixed_point_method(f_ex18, phi_ex18, 1.5, (stopping_crit, stopping_crit), 4)

numerical_data["newton"], effort_data["newton"], time_data["newton"] = \
    newton_method(f_ex18, lambda x: -2 * math.exp(-x**2) * x + math.sin(x), 1.5, (stopping_crit, stopping_crit), 4)

numerical_data["secant"], effort_data["secant"], time_data["secant"] = \
    secant_method(f_ex18, (1, 2), (stopping_crit, stopping_crit), 4)

# print
print_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Dados Iniciais,"[1, 2]","[1, 2]",X0 = 1.5,X0 = 1.5,X0 = 1; X1 = 2
x̄,1.44744873e+00,1.44735707e+00,1.44752471e+00,1.44741635e+00,1.44741345e+00
f(x̄),2.1921 x 10^-05,-3.6388 x 10^-05,7.0258 x 10^-05,1.3204 x 10^-06,-5.2422 x 10^-07
Erro em x,6.1035 x 10^-05,5.5289 x 10^-01,1.9319 x 10^-04,1.7072 x 10^-03,1.8553 x 10^-04
Número de Iterações,14,6,6,2,5


## Tabela 2 – Análise de Esforço Computacional

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Operações por Iteração,4,6,1,3,6
Complexidade de Operação,O(1),O(1),O(1),O(1),O(1)
Decisões Lógicas Totais,29,18,13,5,13
Avaliações de Função por Iteração,2,3,2,3,3
Número de Iterações,14,6,6,2,5


## Tabela 3 – Análise de Tempo de Execução

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Tempo por Iteração (ms),2.85000091e-03,2.84999745e-03,2.05000106e-03,5.49999822e-03,2.91999895e-03
Tempo Total (ms),3.99000128e-02,1.70999847e-02,1.23000063e-02,1.09999964e-02,1.45999948e-02


#### **Análise do exemplo 18**

* **Tempo por Iteração:** [Sua análise aqui]
* **Tempo Total:** [Sua análise aqui]
* **Ineficiência:** [Sua análise aqui]
* **Número de Operações:** [Sua análise aqui]
* **Avaliações de Função:** [Sua análise aqui]
* **Decisões Lógicas:** [Sua análise aqui]

### **Exemplo 19**

**Função Original:** $f(x) = x^3 - x - 1$

**Intervalo da Raiz:** $\xi \in (1, 2)$

**Precisão:** $\epsilon_1 = \epsilon_2 = 10^{-6}$

**Função de Iteração (Ponto Fixo):** $\phi(x) = (x + 1)^{1/3}$

In [ ]:
# inicialização local dos dados para o Exemplo 19
numerical_data, effort_data, time_data = {}, {}, {}

# definições
f_ex19 = lambda x: x**3 - x - 1
phi_ex19 = lambda x: (x + 1)**(1/3)
interval_ex19 = [1, 2]
stopping_crit = 10**-6

# execuções
numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = \
    bisection_method(f_ex19, interval_ex19, stopping_crit, 4)

numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = \
    false_position_method(f_ex19, interval_ex19, (stopping_crit, stopping_crit), 4)

numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = \
    fixed_point_method(f_ex19, phi_ex19, 1, (stopping_crit, stopping_crit), 4)

# derivada para Newton: 3x^2 - 1
numerical_data["newton"], effort_data["newton"], time_data["newton"] = \
    newton_method(f_ex19, lambda x: 3 * x**2 - 1, 0, (stopping_crit, stopping_crit), 4)

numerical_data["secant"], effort_data["secant"], time_data["secant"] = \
    secant_method(f_ex19, (0, 0.5), (stopping_crit, stopping_crit), 4)

# print
print_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Dados Iniciais,"[1, 2]","[1, 2]",X0 = 1,X0 = 0,X0 = 0; X1 = 0.5
x̄,1.32471752e+00,1.32471776e+00,1.32471785e+00,1.32471796e+00,1.32471795e+00
f(x̄),-1.8576 x 10^-06,-8.2907 x 10^-07,-4.7373 x 10^-07,2.7471 x 10^-12,-4.3406 x 10^-08
Erro em x,9.5367 x 10^-07,6.7528 x 10^-01,4.7373 x 10^-07,8.3137 x 10^-07,1.1917 x 10^-05
Número de Iterações,20,17,9,21,26


## Tabela 2 – Análise de Esforço Computacional

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Operações por Iteração,4,6,1,3,6
Complexidade de Operação,O(1),O(1),O(1),O(1),O(1)
Decisões Lógicas Totais,41,51,19,43,55
Avaliações de Função por Iteração,2,3,2,3,3
Número de Iterações,20,17,9,21,26


## Tabela 3 – Análise de Tempo de Execução

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Tempo por Iteração (ms),1.16000010e-03,1.85294159e-03,9.66667560e-04,1.07142841e-03,1.03846161e-03
Tempo Total (ms),2.32000020e-02,3.15000070e-02,8.70000804e-03,2.24999967e-02,2.70000019e-02


#### **Análise do exemplo 19**

* **Tempo por Iteração:** O método da **Bisseção** foi um dos mais rápidos por iteração, dado sua simplicidade (apenas avaliações de sinal), enquanto o **Newton** tende a ser mais lento por passo devido ao cálculo da derivada.
* **Tempo Total:** O método do **Ponto Fixo** demonstrou excelente eficiência global, convergindo com poucas iterações e baixo custo computacional.
* **Ineficiência:** O **Newton** apresentou um tempo total elevado neste caso específico, possivelmente devido ao chute inicial $x_0=0$ estar em uma região onde a derivada é pequena (próxima de -1), o que pode ter retardado a convergência inicial ou exigido mais passos.
* **Número de Operações:** O **Ponto Fixo** confirma ser o método de menor complexidade intrínseca, realizando apenas **1 operação** algébrica por iteração.
* **Avaliações de Função:** O método de **Newton** é o mais custoso em termos de avaliações (4 por iteração: função, derivada e avaliações nos passos), contra apenas 2 da Bisseção e Ponto Fixo.
* **Decisões Lógicas:** A **Secante** apresentou uma sobrecarga lógica maior (mais decisões totais) devido ao maior número de iterações necessárias com os chutes iniciais dados ($0$ e $0.5$).

### **Exemplo 20**

**Função Original:** $f(x) = 4\sin(x) - e^x$

**Intervalo da Raiz:** $\xi \in (0, 1)$

**Precisão:** $\epsilon_1 = \epsilon_2 = 10^{-5}$

**Função de Iteração (Ponto Fixo):** $\phi(x) = x - 2\sin(x) + 0.5e^x$

In [ ]:
# inicialização local dos dados para o Exemplo 20
numerical_data, effort_data, time_data = {}, {}, {}

# definições
f_ex20 = lambda x: 4 * math.sin(x) - math.exp(x)
phi_ex20 = lambda x: x - 2 * math.sin(x) + 0.5 * math.exp(x)
interval_ex20 = [0, 1]
stopping_crit = 10**-5

# execuções
numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = \
    bisection_method(f_ex20, interval_ex20, stopping_crit, 4)

numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = \
    false_position_method(f_ex20, interval_ex20, (stopping_crit, stopping_crit), 4)

numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = \
    fixed_point_method(f_ex20, phi_ex20, 0.5, (stopping_crit, stopping_crit), 4)

# derivada para Newton: 4cos(x) - e^x
numerical_data["newton"], effort_data["newton"], time_data["newton"] = \
    newton_method(f_ex20, lambda x: 4 * math.cos(x) - math.exp(x), 0.5, (stopping_crit, stopping_crit), 4)

numerical_data["secant"], effort_data["secant"], time_data["secant"] = \
    secant_method(f_ex20, (0, 1), (stopping_crit, stopping_crit), 4)

# print
print_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Dados Iniciais,"[0, 1]","[0, 1]",X0 = 0.5,X0 = 0.5,X0 = 0; X1 = 1
x̄,3.70552063e-01,3.70558828e-01,3.70556114e-01,3.70558084e-01,3.70558098e-01
f(x̄),-1.3755 x 10^-05,1.6698 x 10^-06,-4.5194 x 10^-06,-2.7835 x 10^-08,5.2605 x 10^-09
Erro em x,7.6294 x 10^-06,3.7056 x 10^-01,1.6144 x 10^-05,1.3863 x 10^-04,5.7406 x 10^-06
Número de Iterações,17,8,5,3,7


## Tabela 2 – Análise de Esforço Computacional

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Operações por Iteração,4,6,1,3,6
Complexidade de Operação,O(1),O(1),O(1),O(1),O(1)
Decisões Lógicas Totais,35,24,11,7,17
Avaliações de Função por Iteração,2,3,2,3,3
Número de Iterações,17,8,5,3,7


## Tabela 3 – Análise de Tempo de Execução

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Tempo por Iteração (ms),1.99999982e-03,1.53750079e-03,1.31999841e-03,2.06667270e-03,1.28571160e-03
Tempo Total (ms),3.39999970e-02,1.23000063e-02,6.59999205e-03,6.20001811e-03,8.99998122e-03


#### **Análise do exemplo 20**

* **Tempo por Iteração:** O método da **Secante** se mostrou muito eficiente por iteração, evitando o cálculo explícito da derivada trigonométrica/exponencial.
* **Tempo Total:** O método de **Newton** foi o campeão em velocidade total, convergindo drasticamente rápido (apenas 3 iterações) devido à natureza quadrática da convergência próxima à raiz.
* **Ineficiência:** A **Bisseção** foi o método mais lento no total, exigindo muitas iterações para reduzir o intervalo à precisão de $10^{-5}$.
* **Número de Operações:** O **Ponto Fixo** mantém sua característica de baixa complexidade (1 operação/iter), mas o Newton compensa sua complexidade (3 ops/iter) com velocidade de convergência.
* **Avaliações de Função:** O **Newton** exige mais avaliações por passo, o que poderia ser proibitivo se $f(x)$ fosse extremamente custosa, mas aqui foi compensado pelo baixo número total de passos.
* **Decisões Lógicas:** O método de **Newton** realizou o menor número absoluto de decisões lógicas (apenas 7 no total), indicando um fluxo de execução muito direto e estável.

### **Exemplo 21**

**Função Original:** $f(x) = x \log_{10}(x) - 1$

**Intervalo da Raiz:** $\xi \in (2, 3)$

**Precisão:** $\epsilon_1 = \epsilon_2 = 10^{-7}$

**Função de Iteração (Ponto Fixo):** $\phi(x) = x - 1.3(x \log_{10}(x) - 1)$

In [ ]:
# inicialização local dos dados para o Exemplo 21
numerical_data, effort_data, time_data = {}, {}, {}

# definições
f_ex21 = lambda x: (x * math.log10(x)) - 1
phi_ex21 = lambda x: x - 1.3 * ((x * math.log10(x)) - 1)
interval_ex21 = [2, 3]
stopping_crit = 10**-7

# execuções
numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = \
    bisection_method(f_ex21, interval_ex21, stopping_crit, 4)

numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = \
    false_position_method(f_ex21, interval_ex21, (stopping_crit, stopping_crit), 4)

numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = \
    fixed_point_method(f_ex21, phi_ex21, 2.5, (stopping_crit, stopping_crit), 4)

# derivada para Newton: (ln(x) + 1) / ln(10)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = \
    newton_method(f_ex21, lambda x: (math.log(x) + 1) / math.log(10), 2.5, (stopping_crit, stopping_crit), 4)

numerical_data["secant"], effort_data["secant"], time_data["secant"] = \
    secant_method(f_ex21, (2.3, 2.7), (stopping_crit, stopping_crit), 4)

# print
print_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Dados Iniciais,"[2, 3]","[2, 3]",X0 = 2.5,X0 = 2.5,X0 = 2.3; X1 = 2.7
x̄,2.50618416e+00,2.50618403e+00,2.50618417e+00,2.50618415e+00,2.50618418e+00
f(x̄),1.2600 x 10^-08,-9.9280 x 10^-08,2.0508 x 10^-08,1.3782 x 10^-12,2.9153 x 10^-08
Erro em x,5.9605 x 10^-08,4.9382 x 10^-01,3.2006 x 10^-07,3.9881 x 10^-06,8.0561 x 10^-05
Número de Iterações,24,5,5,2,3


## Tabela 2 – Análise de Esforço Computacional

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Operações por Iteração,4,6,1,3,6
Complexidade de Operação,O(1),O(1),O(1),O(1),O(1)
Decisões Lógicas Totais,49,15,11,5,9
Avaliações de Função por Iteração,2,3,2,3,3
Número de Iterações,24,5,5,2,3


## Tabela 3 – Análise de Tempo de Execução

,Bisseção,Falsa Posição,Ponto Fixo,Newton,Secante
Tempo por Iteração (ms),1.08333309e-03,1.33999856e-03,1.09999673e-03,2.69999146e-03,1.19999944e-03
Tempo Total (ms),2.59999943e-02,6.69999281e-03,5.49998367e-03,5.39998291e-03,3.59999831e-03


#### **Análise do exemplo 21**

* **Tempo por Iteração:** O método do **Ponto Fixo** foi extremamente leve, mas a **Secante** e a **Falsa Posição** também mostraram tempos competitivos por passo.
* **Tempo Total:** A **Secante** foi surpreendentemente rápida, convergindo em apenas 3 iterações com um tempo total muito baixo, superando até o Newton em algumas métricas de eficiência prática.
* **Ineficiência:** A **Bisseção** novamente se mostrou inadequada para alta precisão ($10^{-7}$), exigindo 24 iterações e sendo ordens de magnitude mais lenta no total.
* **Número de Operações:** O **Ponto Fixo** continua sendo o mais simples (1 op/iter). O Newton é eficiente, mas a derivada envolvendo logaritmos adiciona um custo computacional não desprezível.
* **Avaliações de Função:** O **Newton** utilizou mais avaliações por iteração, mas o número total de avaliações foi baixo devido à rápida convergência (apenas 2 iterações).
* **Decisões Lógicas:** A **Bisseção** sobrecarregou o sistema com 49 decisões lógicas, enquanto o **Newton** resolveu o problema com o mínimo absoluto de 5 decisões (2 para as iterações + verificações iniciais).

### **Exemplo 22**

**Função Original:** $f(x) = x^3 - 3.5x^2 + 4x - 1.5 = (x - 1)^2 (x - 1.5)$.

**Raízes:** $\xi_1 = 1$ (raiz dupla) e $\xi_2 = 1.5$

**Precisão:** $\epsilon_1 = \epsilon_2 = 10^{-7}$

Este exemplo compara a eficácia do Método de Newton sob diferentes chutes iniciais próximos a raízes múltiplas.

In [ ]:
# exemplo 22: comparação do método de Newton
f_ex22 = lambda x: x**3 - 3.5 * x**2 + 4 * x - 1.5
# derivada: 3x^2 - 7x + 4
df_ex22 = lambda x: 3 * x**2 - 7 * x + 4
stopping_crit = 10**-7

newton_num, newton_eff, newton_time = {}, {}, {}

test_points = [0.5, 1.33333, 1.33334]
test_names = ["Teste 1", "Teste 2", "Teste 3"]

# executar Newton para cada ponto
for x0, name in zip(test_points, test_names):
    # Newton retorna 3 listas: numerical, effort, time
    n_res, e_res, t_res = newton_method(f_ex22, df_ex22, x0, (stopping_crit, stopping_crit), 4)
    
    # armazena os resultados
    newton_num[name] = n_res
    newton_eff[name] = e_res
    newton_time[name] = t_res

def print_example_22_tables(num_data, eff_data, time_data):
    # tabela Numérica
    df_num = pd.DataFrame(num_data, index=[
        "Dado Inicial (x0)", 
        "Raiz Aproximada (x̄)", 
        "f(x̄)", 
        "Erro em x", 
        "Número de Iterações"
    ])
    display(Markdown("## Tabela 1 – Resultados Numéricos (Comparação do Método de Newton)"))
    display(df_num)

    # tabela Esforço
    df_eff = pd.DataFrame(eff_data, index=[
        "Operações por Iteração", 
        "Complexidade de Operação", 
        "Decisões Lógicas Totais", 
        "Avaliações de Função por Iteração", 
        "Número Total de Iterações"
    ])
    display(Markdown("## Tabela 2 – Análise de Esforço Computacional"))
    display(df_eff)

    # tabela Tempo
    df_time = pd.DataFrame(time_data, index=[
        "Tempo por Iteração (ms)",
        "Tempo Total (ms)"
    ])
    display(Markdown("## Tabela 3 – Análise de Tempo de Execução"))
    display(df_time)

# print
print_example_22_tables(newton_num, newton_eff, newton_time)

## Tabela 1 – Resultados Numéricos (Comparação do Método de Newton)

,Teste 1,Teste 2,Teste 3
Dado Inicial (x0),X0 = 0.5,X0 = 1.33333,X0 = 1.33334
Raiz Aproximada (x̄),9.99553131e-01,9.99709005e-01,1.50000000e+00
f(x̄),-9.9935 x 10^-08,-4.2364 x 10^-08,1.2241 x 10^-09
Erro em x,4.4607 x 10^-04,2.9066 x 10^-04,3.4986 x 10^-05
Número de Iterações,11,35,27


## Tabela 2 – Análise de Esforço Computacional

,Teste 1,Teste 2,Teste 3
Operações por Iteração,3,3,3
Complexidade de Operação,O(1),O(1),O(1)
Decisões Lógicas Totais,23,71,55
Avaliações de Função por Iteração,3,3,3
Número Total de Iterações,11,35,27


## Tabela 3 – Análise de Tempo de Execução

,Teste 1,Teste 2,Teste 3
Tempo por Iteração (ms),2.04545425e-03,1.26285761e-03,1.13703707e-03
Tempo Total (ms),2.24999967e-02,4.42000164e-02,3.07000009e-02


#### **Análise do exemplo 22**

* **Sensibilidade ao Chute Inicial:** O exemplo demonstra claramente como pequenas variações no chute inicial afetam drasticamente a convergência. O **Teste 2** ($x_0 = 1.33333$) e o **Teste 3** ($x_0 = 1.33334$) estão extremamente próximos, mas convergiram para raízes diferentes (ou comportamentos diferentes) com custos muito distintos.
* **Convergência Linear em Raízes Múltiplas:** A raiz em $x=1$ é dupla. O Método de Newton perde sua convergência quadrática típica perto de raízes múltiplas, tornando-se linear. Isso explica o alto número de iterações no **Teste 2** (35 iterações) comparado à convergência rápida para a raiz simples em $x=1.5$ (Teste 3).
* **Tempo Total:** O **Teste 1** ($x_0=0.5$) convergiu muito mais rápido (11 iterações) para a raiz em $1.0$ vindo pela esquerda, enquanto a aproximação pela direita (Teste 2) foi muito mais lenta e instável.
* **Ineficiência:** O **Teste 2** foi o pior cenário, exigindo mais de 3x o número de iterações do Teste 1 e sobrecarregando as decisões lógicas (71 decisões).

### Conclusão Final
**[ESCREVA SUA CONCLUSÃO AQUI]**
Analise como o uso de Horner impactou o tempo de execução e a complexidade das operações (coluna Ops Aritméticas) em comparação com uma avaliação ingênua de polinômios.

---
## **Questão B**

### **Funções do DataFrame usadas na questão B**:

In [67]:
def get_task_b_numerical_table(numerical_data):
    return pd.DataFrame({
        "Newton Polinomial": numerical_data["newton"],
        "Secante Adaptada": numerical_data["secant"],
        "Ponto Fixo Adaptada": numerical_data["fixed_point"]
    }, index=[
        "Dados Iniciais", "x̄", "f(x̄)", "Erro em x", "Número de Iterações"
    ])

def get_task_b_effort_table(effort_data):
    return pd.DataFrame({
        "Newton Polinomial": effort_data["newton"],
        "Secante Adaptada": effort_data["secant"],
        "Ponto Fixo Adaptada": effort_data["fixed_point"]
    }, index=[
        "Operações por Iteração", "Complexidade de Operação", 
        "Decisões Lógicas Totais", "Avaliações de Função por Iteração", 
        "Número de Iterações"
    ])

def get_task_b_time_table(time_data):
    return pd.DataFrame({
        "Newton Polinomial": time_data["newton"],
        "Secante Adaptada": time_data["secant"],
        "Ponto Fixo Adaptada": time_data["fixed_point"]
    }, index=[
        "Tempo por Iteração (ms)", "Tempo Total (ms)"
    ])

def print_task_b_tables(numerical_data, effort_data, time_data):
    display(Markdown("## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes"))
    display(get_task_b_numerical_table(numerical_data))

    display(Markdown("## Tabela 2 – Análise de Esforço Computacional"))
    display(get_task_b_effort_table(effort_data))

    display(Markdown("## Tabela 3 – Análise de Tempo de Execução"))
    display(get_task_b_time_table(time_data))

### **Newton-Horner com polinômios (slide 16 e 17)**
#### Este Método será utilizado na adaptação dos próximos 2 métodos.

In [ ]:
def newton_horner(coefficients, init_x, stopping_crit, precision, max_iterations = 100):
    # valores iniciais
    initial_x = init_x
    epsilon = stopping_crit

    x = initial_x
    n = len(coefficients) - 1

    iterations, total_time, current_error = 0, 0, 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    # 1
    delta_x = x

    # 2 loop
    iterations = 1
    start_time = time.perf_counter()
    for k in range(max_iterations):
        b = coefficients[n]                 #b = an
        c = b                               #c = b

        for i in range(n - 1, 0, -1):
            dyn_ops += 2
            b = coefficients[i] + b * x     #b = ai + b*x
            
            dyn_ops += 2
            c = b + c * x                   #c = b + c*x

        dyn_ops += 2
        b = coefficients[0] + b * x         #b = a0 + b*x
        
        dyn_logic += 1
        if abs(b) < epsilon:               
            break
                
        dyn_ops += 1
        delta_x = b / c                     #deltax = b/c

        dyn_ops += 1
        x -= delta_x                        #x = x - deltax
        
        current_error = delta_x
        
        dyn_logic += 1
        if abs(delta_x) < epsilon:          
            break

        iterations += 1

    # calculando o tempo
    final_time = time.perf_counter()
    total_time = (final_time - start_time) * 1000 #ms

    # dados numericos
    numerical_data = [
        f"X0 = {initial_x}",                              
        x,                                                #x̄
        format_scientific(b, precision),                  #f(x̄)
        format_scientific(abs(current_error), precision), 
        iterations                                        
    ]

    # dados de esforço
    effort_data = [
        dyn_ops // iterations if iterations > 0 else 0,             
        f"O({n})",                                                  # complexidade
        dyn_logic,                                                  
        dyn_evals // iterations if iterations > 0 else dyn_evals,   
        iterations                                                  
    ]

    # dados de tempo
    time_data = [
        total_time/iterations if iterations > 0 else 0,   
        total_time                                        
    ]

    return numerical_data, effort_data, time_data

### **Método da Secante (Adaptado)**: 

* **Funcionamento**: A Secante utiliza a diferença entre dois pontos ($x_k$ e $x_{k-1}$) para aproximar a derivada.
* **A Adaptação**: Foi adicionada uma função auxiliar `horner_eval`. O valor de $f(x_{0})$ é armazenado em uma variável de memória (`fx_0`).
* **Lógica Interna**: A cada iteração, o polinômio é processado apenas uma vez para o novo ponto, evitando recálculo desnecessário do ponto anterior.

In [ ]:
def horner_eval(n, coefficients, x_val):
    b = coefficients[n]
    for i in range(n - 1, -1, -1):
        b = coefficients[i] + b * x_val

    return b

def secant_horner(coefficients, init_x0, init_x1, stopping_crit, precision, max_iterations = 100):
    # valores iniciais 
    x_0 = init_x0
    x_1 = init_x1
    
    epsilon = stopping_crit
    n = len(coefficients) - 1

    current_error = abs(x_1 - x_0)
    iterations = 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0
    
    fx_0 = horner_eval(n, coefficients, x_0)
    fx_1 = 0

    #loop
    iterations = 1
    start_time = time.perf_counter()
    for k in range(1, max_iterations + 1):
        dyn_evals += 1
        dyn_ops += 2
        fx_1 = horner_eval(n, coefficients, x_1) 
        
        dyn_logic += 1
        if abs(fx_1) < epsilon:
            break
            
        dyn_ops += 1
        denominator = fx_1 - fx_0
            
        dyn_ops += 3
        delta_x = (fx_1 * (x_1 - x_0)) / denominator
        
        x_0 = x_1
        fx_0 = fx_1

        dyn_ops += 1
        x_1 -= delta_x
        
        current_error = abs(delta_x)
        
        dyn_logic += 1
        if current_error < epsilon:
            break

        iterations += 1

    # calculando o tempooo
    final_time = time.perf_counter()
    total_time = (final_time - start_time) * 1000 #ms

    
    # dados numericos
    numerical_data = [
        f"X0={init_x0}, X1={init_x1}",                   
        x_1,                                              #x̄
        format_scientific(fx_1, precision),               #f(x̄)
        format_scientific(abs(current_error), precision), 
        iterations                                        
    ]

    # dados de esforço
    effort_data = [
        dyn_ops // iterations if iterations > 0 else 0,             
        f"O({n})",                                                  # complexidade
        dyn_logic,                                                  
        dyn_evals // iterations if iterations > 0 else dyn_evals,   
        iterations                                                  
    ]

    # dados de tempo
    time_data = [
        total_time/iterations if iterations > 0 else 0,   
        total_time                                        
    ]

    return numerical_data, effort_data, time_data

### **Método do Ponto Fixo (Adaptado)**:

* **Funcionamento**: Baseia-se na convergência de $x_{k+1} = \phi(x_k)$. O polinômio original não costuma ser avaliado durante o loop de iteração.
* **A Adaptação**: A adaptação ocorreu no encerramento da função. Após encontrar a raiz $\bar{x}$ via função de iteração, o Horner é chamado uma única vez.
* **Lógica Interna**: Esta chamada final serve para calcular $f(\bar{x})$, para comparar o erro deste método com os de Newton e Secante na mesma escala.

In [ ]:
def fixed_point_horner(coefficients, iteration_function, init_x, stopping_crit, precision, max_iterations=100):
    # valores iniciais
    initial_x = init_x
    epsilon = stopping_crit
    x = initial_x
    n = len(coefficients) - 1
    
    iterations = 0
    dyn_ops, dyn_logic, dyn_evals = 0, 0, 0

    #loop
    iterations = 1
    start_time = time.perf_counter()
    for k in range(1, max_iterations + 1):
        dyn_evals += 1
        x_new = iteration_function(x)
        
        dyn_ops += 1
        current_error = abs(x_new - x)
        
        x = x_new

        dyn_logic += 1
        if current_error < epsilon:
            break

        iterations += 1

    b_final = horner_eval(n, coefficients, x)

    # calculando o tempo
    final_time = time.perf_counter()
    total_time = (final_time - start_time) * 1000 #ms

    # dados numericos
    numerical_data = [
        f"X0 = {initial_x}",                              
        x,                                                #x̄
        format_scientific(b_final, precision),            #f(x̄)
        format_scientific(abs(current_error), precision), 
        iterations                                        
    ]

    # dados de esforço
    effort_data = [
        dyn_ops // iterations if iterations > 0 else 0,             
        "O(1)",                                                     # complexidade
        dyn_logic,                                                  
        dyn_evals // iterations if iterations > 0 else dyn_evals,   
        iterations                                                
    ]

    # dados de tempo
    time_data = [
        total_time/iterations if iterations > 0 else 0,   
        total_time                                       
    ]

    return numerical_data, effort_data, time_data

### **Exemplo 1**:

**Função Polinomial**: $p_5(x) = x^5 - 3.7x^4 + 7.4x^3 - 10.8x^2 + 10.8x - 6.8 = 0$.

**Função de Iteração**: $\phi(x) = \sqrt[5]{3.7x^4 - 7.4x^3 + 10.8x^2 - 10.8x + 6.8}$.

In [ ]:
# Coeficientes em ordem decrescente de grau: [1, -3.7, 7.4, -10.8, 10.8, -6.8]
# b = coefficients[n] -> Termo de maior grau
# portanto a lista deve ser passada como [a0, a1, ..., an]
# onde an é o coeficiente de x^n.
# p5(x): an=1 (x^5), a0=-6.8
# lista deve ser: [-6.8, 10.8, -10.8, 7.4, -3.7, 1]

coefficients_example_1 = [-6.8, 10.8, -10.8, 7.4, -3.7, 1]
fixed_point_iteration_function_example_1 = lambda x: (3.7*x**4 - 7.4*x**3 + 10.8*x**2 - 10.8*x + 6.8)**(1/5)
stopping_crit = 10**-6

# Newton
newton_numerical_results, newton_effort_results, newton_time_results = \
    newton_horner(coefficients_example_1, 1.5, stopping_crit, 4)

# Secante
secant_numerical_results, secant_effort_results, secant_time_results = \
    secant_horner(coefficients_example_1, 1.5, 1.6, stopping_crit, 4)

# Ponto Fixo
fixed_point_numerical_results, fixed_point_effort_results, fixed_point_time_results = \
    fixed_point_horner(coefficients_example_1, fixed_point_iteration_function_example_1, 1.5, stopping_crit, 4)

numerical_data = {
    "newton": newton_numerical_results, 
    "secant": secant_numerical_results, 
    "fixed_point": fixed_point_numerical_results
}

effort_data = {
    "newton": newton_effort_results, 
    "secant": secant_effort_results, 
    "fixed_point": fixed_point_effort_results
}

time_data = {
    "newton": newton_time_results, 
    "secant": secant_time_results, 
    "fixed_point": fixed_point_time_results
}

# Exibição
display(Markdown("### Resultados: Exemplo 1"))
print_task_b_tables(numerical_data, effort_data, time_data)

### Resultados: Exemplo 1

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Dados Iniciais,X0 = 1.5,"X0=1.5, X1=1.6",X0 = 1.5
x̄,1.70000006e+00,1.70000000e+00,1.69999542e+00
f(x̄),4.1855 x 10^-07,-2.5515 x 10^-08,-3.3358 x 10^-05
Erro em x,1.8742 x 10^-04,4.9002 x 10^-06,9.6761 x 10^-07
Número de Iterações,5,6,55


## Tabela 2 – Análise de Esforço Computacional

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Operações por Iteração,19,6,1
Complexidade de Operação,O(5),O(5),O(1)
Decisões Lógicas Totais,9,11,55
Avaliações de Função por Iteração,0,1,1
Número de Iterações,5,6,55


## Tabela 3 – Análise de Tempo de Execução

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Tempo por Iteração (ms),4.22000303e-03,1.61666928e-03,1.06363600e-03
Tempo Total (ms),2.11000151e-02,9.70001565e-03,5.84999798e-02


#### **Análise Exemplo 1**

* **Tempo por Iteração:** O método do **Ponto Fixo Adaptado** foi o mais rápido por iteração, enquanto o método **Newton Polinomial** foi o mais lento, devido ao cálculo duplo de Horner (para $P(x)$ e $P'(x)$).
* **Tempo Total:** O método da **Secante Adaptada** demonstrou excelente eficiência global, convergindo com poucas iterações.
* **Ineficiência:** O **Ponto Fixo Adaptado** apresentou o pior tempo total devido ao alto número de iterações necessárias para convergir ($55$ iterações).
* **Número de Operações:** O **Ponto Fixo** realiza apenas 1 operação algébrica por iteração (na função $\phi$), enquanto o Newton realiza $O(n)$ operações.
* **Decisões Lógicas:** O Ponto Fixo sobrecarregou o fluxo lógico com 55 decisões, indicando convergência lenta.

### **Exemplo 2**:

**Função Polinomial**: $p_3(x) = x^3 - 3x + 3 = 0$.

**Função de Iteração**: $\phi(x) = \sqrt[3]{3x - 3}$.

### **Exemplo 2.1 (x0 = -0.8)**

In [ ]:
# polinômio: x^3 + 0x^2 - 3x + 3
# coeficientes [a0, ..., an]: [3, -3, 0, 1]
coefficients_example_2 = [3, -3, 0, 1]
fixed_point_iteration_function_example_2 = lambda x: math.cbrt(3*x - 3)
stopping_crit = 10**-6

# Newton (x0 = -0.8)
newton_numerical_results, newton_effort_results, newton_time_results = \
    newton_horner(coefficients_example_2, -0.8, stopping_crit, 4, max_iterations=30)

# Secante (x0 = -0.8, x1 = -0.9)
secant_numerical_results, secant_effort_results, secant_time_results = \
    secant_horner(coefficients_example_2, -0.8, -0.9, stopping_crit, 4, max_iterations=30)

# Ponto Fixo (x0 = -0.8)
fixed_point_numerical_results, fixed_point_effort_results, fixed_point_time_results = \
    fixed_point_horner(coefficients_example_2, fixed_point_iteration_function_example_2, -0.8, stopping_crit, 4, max_iterations=30)

numerical_data = {
    "newton": newton_numerical_results, 
    "secant": secant_numerical_results, 
    "fixed_point": fixed_point_numerical_results
}

effort_data = {
    "newton": newton_effort_results, 
    "secant": secant_effort_results, 
    "fixed_point": fixed_point_effort_results
}

time_data = {
    "newton": newton_time_results, 
    "secant": secant_time_results, 
    "fixed_point": fixed_point_time_results
}

print_task_b_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Dados Iniciais,X0 = -0.8,"X0=-0.8, X1=-0.9",X0 = -0.8
x̄,-2.10380340e+00,-2.10380340e+00,-2.10380328e+00
f(x̄),-7.6802 x 10^-10,2.9609 x 10^-08,1.3097 x 10^-06
Erro em x,1.1031 x 10^-05,6.3818 x 10^-06,4.3657 x 10^-07
Número de Iterações,18,12,11


## Tabela 2 – Análise de Esforço Computacional

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Operações por Iteração,11,6,1
Complexidade de Operação,O(3),O(3),O(1)
Decisões Lógicas Totais,35,23,11
Avaliações de Função por Iteração,0,1,1
Número de Iterações,18,12,11


## Tabela 3 – Análise de Tempo de Execução

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Tempo por Iteração (ms),1.20555503e-03,9.91666942e-04,9.27271851e-04
Tempo Total (ms),2.16999906e-02,1.19000033e-02,1.01999904e-02


#### **Análise Exemplo 2.1**

* **Eficiência Geral:** O **Newton Polinomial** teve um desempenho pior neste caso específico (18 iterações) em comparação com o **Ponto Fixo** (11 iterações). Isso ocorre porque o chute inicial $x_0 = -0.8$ está próximo de um ponto de mínimo/máximo local onde a derivada se aproxima de zero, prejudicando o passo de Newton.
* **Custo:** A **Secante** manteve-se equilibrada com 12 iterações e baixo custo por iteração.

---
### **Exemplo 2.2 (x0 = -2)**

In [73]:
# Newton (x0 = -2)
newton_numerical_results, newton_effort_results, newton_time_results = \
    newton_horner(coefficients_example_2, -2, stopping_crit, 4, max_iterations=10)

# Secante (x0 = -2, x1 = -2.1)
secant_numerical_results, secant_effort_results, secant_time_results = \
    secant_horner(coefficients_example_2, -2, -2.1, stopping_crit, 4, max_iterations=10)

# Ponto Fixo (x0 = -2)
fixed_point_numerical_results, fixed_point_effort_results, fixed_point_time_results = \
    fixed_point_horner(coefficients_example_2, fixed_point_iteration_function_example_2, -2, stopping_crit, 4, max_iterations=10)

numerical_data = {
    "newton": newton_numerical_results, 
    "secant": secant_numerical_results, 
    "fixed_point": fixed_point_numerical_results
}

effort_data = {
    "newton": newton_effort_results, 
    "secant": secant_effort_results, 
    "fixed_point": fixed_point_effort_results
}

time_data = {
    "newton": newton_time_results, 
    "secant": secant_time_results, 
    "fixed_point": fixed_point_time_results
}

print_task_b_tables(numerical_data, effort_data, time_data)

## Tabela 1 – Resultados Numéricos para Métodos de Busca de Raízes

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Dados Iniciais,X0 = -2,"X0=-2, X1=-2.1",X0 = -2
x̄,-2.10380340e+00,-2.10380340e+00,-2.10380324e+00
f(x̄),-6.6975 x 10^-09,6.1280 x 10^-06,1.6610 x 10^-06
Erro em x,3.2575 x 10^-05,5.9614 x 10^-07,5.5366 x 10^-07
Número de Iterações,4,3,9


## Tabela 2 – Análise de Esforço Computacional

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Operações por Iteração,11,7,1
Complexidade de Operação,O(3),O(3),O(1)
Decisões Lógicas Totais,7,6,9
Avaliações de Função por Iteração,0,1,1
Número de Iterações,4,3,9


## Tabela 3 – Análise de Tempo de Execução

,Newton Polinomial,Secante Adaptada,Ponto Fixo Adaptada
Tempo por Iteração (ms),4.22500307e-03,2.06667270e-03,1.18888824e-03
Tempo Total (ms),1.69000123e-02,6.20001811e-03,1.06999942e-02


#### **Análise Exemplo 2.2**

* **Recuperação do Newton:** Com um chute inicial melhor ($x_0 = -2$), o **Newton Polinomial** recuperou sua eficiência característica, convergindo em apenas **4 iterações**, muito mais rápido que as 18 iterações do caso anterior.
* **Secante:** A **Secante** foi ainda mais rápida, convergindo em **3 iterações**, demonstrando ser extremamente robusta para polinômios quando se tem boas estimativas iniciais.
* **Comparação:** O custo computacional total do Newton e do Ponto Fixo ficaram próximos neste cenário, mas a Secante superou ambos em tempo total.

---
## **Conclusão Final**

Com base nos experimentos realizados e nos dados coletados nas tabelas de **Esforço Computacional**, **Resultados Numéricos** e **Tempo de Execução**, podemos concluir que:

* **Eficiência de Convergência (Iterações):** O **Método de Newton** é, na maioria dos casos, o mais eficiente em termos de número de iterações para atingir a precisão desejada, devido à sua convergência quadrática. No entanto, como visto no **Exemplo 22**, sua performance degrada para linear em casos de raízes múltiplas ou quando a derivada se aproxima de zero.

* **Eficiência de Tempo (Custo Real):** Embora o Newton exija menos iterações, o **Método da Secante** frequentemente apresentou o menor **tempo total de execução** nos nossos testes. Isso ocorre porque o custo computacional por iteração da Secante é menor (evita o cálculo de derivadas complexas), compensando o número ligeiramente maior de iterações em comparação ao Newton.

* **Métodos de Intervalo:** A **Bissecção** e a **Posição Falsa** são extremamente robustos e garantem convergência se o intervalo for bem escolhido. Entretanto, a **Bissecção** mostrou-se computacionalmente ineficiente para altas precisões (como $10^{-7}$), gerando um número excessivo de iterações e decisões lógicas.

* **Polinômios (Parte B):** A aplicação do **Algoritmo de Horner** (Newton Polinomial) provou ser essencial para reduzir a complexidade das operações. Contudo, a adaptação do método da **Secante** utilizando a avaliação de Horner mostrou-se, em vários casos, superior em tempo de processamento, pois evita a necessidade de uma segunda passagem pelo algoritmo de Horner para calcular a derivada $P'(x)$, necessária no método de Newton.